# `DocLayNet` Linknet Architecture
**Author**: Juan Pablo Triana Martinez.

The following notebook contains all of the `torch.nn` codes related  to recreate a `linknet-resnet` architecture!

We will dive deep into the `resnet` paper, while also maintaining all of its components, which are:
1. Encoder blocks
2. Decoder Blocks
3. Full-Conv Block

## Architecture.
We will reference the `linknet-resnet` paper in order to retrieve the padding, strides, and outputs at each layer.

**Input Shape:** Our input shape will be assumed to be `(3 * 1024 * 1024)`. So we can get Batches of size `(B * 3 * 1024 * 1024)`

<div style="text-align: center;">
    <img src="../docs/linknet_resnet18_architecture.png" width="550">
</div>

It seems the `linknet-resnet` architecture depends on 4 different components:
-  An initial input block, we will call it `linknet_stem`.
- A bunch of encoder blocks, let's call these ones `linknet_encoders`
- A bunch of decoder blocks, we will follow original names: `linknet_decoders`.
- An the last one appears to be a reconstruction neural network block, the name will be the `linknet_reconstructer`.

But what are the `encoder` and `decoder` blocks look like?

<div style="text-align: center;">
    <img src="../docs/linknet_resnet18_encoder-block.png" width="350" style="margin-right: 10px;">
    <img src="../docs/linknet_resnet18_decoder-block.png" width="350">
</div>

Interesting.... these are blocks that we have seen from deep learning convolutional network resnet!, paper in the following link: https://arxiv.org/abs/1512.03385

So which are the dimensions of **n** and **m** for each layer?:

<img src="../docs/linknet_resnet18_inputs-outputs.png" width="650">

Here, the following is true:
- *n* refers to the number of input channels in the convolution.
- *m* refers to the number of output channels in the convoultion.
- *full-conv* refers to **Convolution Transpose 2D** or **Convolution 2D upsample**.
- The *2* or */2* refers to how much the data has been upsampled or downsampled, by a factor of 2 or divided by 2.
- When there is no number at the end **it means it preserved its dimensions**, meaning there was **padding = 1**.
- In the paper, each convolution layer was followed by a **batch normalization layer**, and a **relu activation function**.


## 1. Encoder Stem.

The first block to create will be the one who will receive our `torch.Tensor` of images (i.e: sizes of multiple of 32 to ensure linknet-resnet works). 

<img src="../docs/linknet_resnet18_stem.png" width="250">

This block of neural networks layers would be composed of the following:
* Using `torch.nn`, use `nn.Conv2d` with the following parameters to perform convolutions:
    - kernel size `(7, 7)`.
    - input channels = 3.
    - output channels = 64.
    - stride = 2. Due to the `/2`, which downsamples the image to half the size.
    - **Padding  = 3**, done for `(7, 7)` initial kernel with `resnet` architecture.
* Using `torch.nn`, use `nn.Maxpool2d` with the following parameters to perform the 2D transformation layer:
    - kernel size `(3, 3)`
    - stride = 2. Due to the `/2`, which downsamples the image to half the size.
    - **Padding = 1**, done for `(3, 3)` kernels with `resnet` architecture.

Where do we get the sources and what kinds of paddings are apprpiate? well the sources are the original paper and this cool nice website where they implement the `resnet` for PyTorch, TensorFlow, and Jax.
* `Resnet` paper link: https://arxiv.org/pdf/1512.03385
* `Resent` code implementations: https://d2l.ai/chapter_convolutional-modern/resnet.html

The typical paddings values we found were the following:
- `(3 * 3)` Convolution contains Padding = 1 (Same padding).
- `(1 * 1)` Convolution contains Padding = 0 (Valid padding).
- `(7 * 7)` Convolution contains Padding = 3 (Resnet 50/101/152 Initial Layer).


In [2]:
# Let's import all necessary modules for linknet architecture
import torch.nn as nn
import torch

In [3]:
# Let's import all necessary modules for linknet architecture
class LinknetStem(nn.Module):
    '''
    Class that will define the stem of the linknet architecture

    Args:
        m (int): Number of input channels from linknet paper.
        n (int): Number of output channels from linknet paper.
    '''

    def __init__(self, m:int = 3, n:int = 64) -> None:
        super().__init__()
        # Let's define the nn.Sequential module
        self.linknet_stem = nn.Sequential(
            nn.Conv2d(in_channels=m, out_channels=n, kernel_size=(7, 7), stride= (2, 2), padding=(3, 3), bias=False),
            nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        )
    
    def forward(self, x) -> torch.Tensor:
        return self.linknet_stem(x)
    

### 1.1 Summary info of `LinknetStem`

In [4]:
from torchinfo import summary
# Let's create an instance of the LinknetStem Block
linknet_stem = LinknetStem(m = 3, n = 64)

summary(model=linknet_stem,
        input_size=(1, 3, 1024, 1024))

Layer (type:depth-idx)                   Output Shape              Param #
LinknetStem                              [1, 64, 256, 256]         --
├─Sequential: 1-1                        [1, 64, 256, 256]         --
│    └─Conv2d: 2-1                       [1, 64, 512, 512]         9,408
│    └─BatchNorm2d: 2-2                  [1, 64, 512, 512]         128
│    └─ReLU: 2-3                         [1, 64, 512, 512]         --
│    └─MaxPool2d: 2-4                    [1, 64, 256, 256]         --
Total params: 9,536
Trainable params: 9,536
Non-trainable params: 0
Total mult-adds (Units.GIGABYTES): 2.47
Input size (MB): 12.58
Forward/backward pass size (MB): 268.44
Params size (MB): 0.04
Estimated Total Size (MB): 281.06

## 2. Encoder blocks

<div style="text-align: center;">
    <img src="../docs/linknet_resnet18_encoder-block.png" width="350" style="margin-right: 10px;">
    <img src="../docs/linknet_resnet18_inputs-outputs.png" width="350">
</div>

The input dimensions are the following, for each encoder block, the dimensions would go by `//2`, and the number of output channels increase based on the table 
:
```python
x_input -> (B * 3 * 1024 * 1024)
x_input -> linknet_stem -> (B * 64 * 256 * 256)
out_linknet_stem -> encoder_1 -> (B * 64 * 128 * 128)
out_linknet_encoder_1 -> encoder_2 -> (B * 128 * 64 * 64)
out_linknet_encoder_2 -> encoder_3 -> (B * 256 * 32 * 32)
out_linknet_encoder_3 -> encoder_4 -> (B * 512 * 16 * 16)
```


### 2.1 First two layers of `encoder` with skip connection.
For the encoder blocks, we are gonna model the following:

- The first convolution layer is of kernel size = `(3, 3)`, stride = `(2, 2)` and padding = `1`.
- The second convolution layer is of kernel size = `(3, 3)`, stride = `(1, 1)`, and padding = `1`

But how do we simulate the skip connections... the following code will explain, we will be modelling the `encoder_2` layer, since the number of input and output channels are completely different.

In [5]:
# Simulation of two convolutional blocks with residual connection
m = 64
n = 128

# Simulate encoder_2 block, the first two layers
convs_block_2_1_test = nn.Sequential(
    nn.Conv2d(in_channels = m, out_channels= n, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False),
    nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
    nn.ReLU())

convs_block_2_2_test = nn.Sequential(
    nn.Conv2d(in_channels = n, out_channels=n, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False),
    nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
    nn.ReLU()
)

# Simulate input tensor from linknet stem + encoder_1
input_tensor = torch.randn((1, 64, 128, 128))

# Set to eval mode to enable inference mode
with torch.inference_mode():
    out_tensor_1:torch.Tensor = convs_block_2_1_test(input_tensor)
    out_tensor_2:torch.Tensor = convs_block_2_2_test(out_tensor_1)

# Let's take a look into the shapes
print(f"Input Tensor shape from linknet stem + encoder_1 {input_tensor.shape}")
print(f"Output Tensor shape from encoder_2, first layer: {out_tensor_1.shape}")
print(f"Output Tensor Shape from encoder_2, second layer: {out_tensor_2.shape}")

# To do the residulial connection, we need to project the input tensor to have the 
skip_connection_1 = nn.Conv2d(in_channels=m, out_channels = n, kernel_size = (1, 1), stride = (2, 2), padding=(0, 0), bias=False)
with torch.inference_mode():
    proj_skip_connection = skip_connection_1(input_tensor)
out_tensor_3 = out_tensor_2 + proj_skip_connection
print(f"Output tensor skip connection projection shape: {proj_skip_connection.shape}")
print(f"Output tensor encoder_2, first two layers + skip connection = {out_tensor_3.shape}")

Input Tensor shape from linknet stem + encoder_1 torch.Size([1, 64, 128, 128])
Output Tensor shape from encoder_2, first layer: torch.Size([1, 128, 64, 64])
Output Tensor Shape from encoder_2, second layer: torch.Size([1, 128, 64, 64])
Output tensor skip connection projection shape: torch.Size([1, 128, 64, 64])
Output tensor encoder_2, first two layers + skip connection = torch.Size([1, 128, 64, 64])


### 2.2 Last two layers of `encoder` with skip connection.

In [6]:
# Let's model now the final bit of the linknet encoder block
convs_block_2_3_test = nn.Sequential(
    nn.Conv2d(in_channels = n, out_channels= n, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False),
    nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
    nn.ReLU())

convs_block_2_4_test = nn.Sequential(
    nn.Conv2d(in_channels = n, out_channels=n, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False),
    nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
    nn.ReLU()
)

# Lets do inference mode and retrieve the tensors
with torch.inference_mode():
    out_tensor_4 = convs_block_2_3_test(out_tensor_3)
    out_tensor_5 = convs_block_2_4_test(out_tensor_4)

# Let's take a look into the shapes
print(f"Encoder_2 first two layers + first skip connection shape: {out_tensor_3.shape}")
print(f"Encoder_2 third layer shape: {out_tensor_4.shape}")
print(f"Encoder_2 fourth layer shape: {out_tensor_5.shape}")

# To do the residulial connection, since they have the same shape, we can add them
out_tensor_6 = out_tensor_5 + out_tensor_3
print(f"Encoder_2 layers 3 and 4 + second skip connection shape: {out_tensor_6.shape}")

Encoder_2 first two layers + first skip connection shape: torch.Size([1, 128, 64, 64])
Encoder_2 third layer shape: torch.Size([1, 128, 64, 64])
Encoder_2 fourth layer shape: torch.Size([1, 128, 64, 64])
Encoder_2 layers 3 and 4 + second skip connection shape: torch.Size([1, 128, 64, 64])


### 2.3 Let's now create the `LinknetEncoderBlock` class with `nn.Module`

In [7]:
class LinknetEncoderBlock(nn.Module):
    '''
    Class that will define a encoder block of the linknet architecture

    Args:
        m (int): number of input channels from linknet paper.
        n (int): number of output channels from linknet paper.
    
    Returns:
        nn.Module: encoder block of linknet architecture.
    '''
    def __init__(self, m:int, n:int) -> None:
        super().__init__()
        # Let's define the first set of convolutional blocks
        self.convs_blocks_1 = nn.Sequential(
            nn.Conv2d(in_channels = m, out_channels= n, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False),
            nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
            nn.ReLU(),
            nn.Conv2d(in_channels = n, out_channels=n, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False),
            nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
            nn.ReLU()
            )
        
        # Let's define the skip connection projection
        self.skip_conn = nn.Conv2d(in_channels=m, out_channels = n, kernel_size = (1, 1), stride = (2, 2), padding=(0, 0), bias=False)

        # Let's define the second set of convolutional blocks.
        self.convs_block_2 = nn.Sequential(
        nn.Conv2d(in_channels = n, out_channels= n, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False),
        nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
        nn.ReLU(),
        nn.Conv2d(in_channels = n, out_channels=n, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False),
        nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
        nn.ReLU())

    def forward(self, x) -> torch.tensor:
        # Pass the first convolutional blocks
        x1 = self.convs_blocks_1(x)

        # Add the residual connection
        x2 = x1 + self.skip_conn(x)

        # Pass the second convolutional blocks
        x3 = self.convs_block_2(x2)

        # Add the final residual connection and output it
        return x3 + x2

### 2.4 Summary of `Encoder_2` block

In [8]:
from torchinfo import summary
# Let's create an instance of the Encoder block #2
linknet_stem = LinknetEncoderBlock(m = 64, n = 128)

summary(model = linknet_stem,
        input_size=(1, 64, 128, 128), # (batch_size, num_channels, height, width)
        col_names = ["input_size", "output_size", "num_params", "trainable"],
        col_width = 20,
        row_settings = ["var_names"]
        )

Layer (type (var_name))                       Input Shape          Output Shape         Param #              Trainable
LinknetEncoderBlock (LinknetEncoderBlock)     [1, 64, 128, 128]    [1, 128, 64, 64]     --                   True
├─Sequential (convs_blocks_1)                 [1, 64, 128, 128]    [1, 128, 64, 64]     --                   True
│    └─Conv2d (0)                             [1, 64, 128, 128]    [1, 128, 64, 64]     73,728               True
│    └─BatchNorm2d (1)                        [1, 128, 64, 64]     [1, 128, 64, 64]     256                  True
│    └─ReLU (2)                               [1, 128, 64, 64]     [1, 128, 64, 64]     --                   --
│    └─Conv2d (3)                             [1, 128, 64, 64]     [1, 128, 64, 64]     147,456              True
│    └─BatchNorm2d (4)                        [1, 128, 64, 64]     [1, 128, 64, 64]     256                  True
│    └─ReLU (5)                               [1, 128, 64, 64]     [1, 128, 64, 64]  

### 2.5 Let's now simulate all passed of the `LinknetEncoderBlock`

In [9]:
from torchinfo import summary
# Let's recreate the sequence of encoder blocks from linknet architecture with linknet stem
test_blocks = nn.Sequential(
    LinknetStem(m = 3, n = 64),
    LinknetEncoderBlock(m = 64, n = 64),
    LinknetEncoderBlock(m = 64, n = 128),
    LinknetEncoderBlock(m = 128, n = 256),
    LinknetEncoderBlock(m = 256, n = 512)
)

summary(model = test_blocks,
        input_size=(1, 3, 1024, 1024), # (batch_size, num_channels, height, width)
        col_names = ["input_size", "output_size", "num_params", "trainable"],
        col_width = 20,
        row_settings = ["var_names"]
        )

Layer (type (var_name))                  Input Shape          Output Shape         Param #              Trainable
Sequential (Sequential)                  [1, 3, 1024, 1024]   [1, 512, 16, 16]     --                   True
├─LinknetStem (0)                        [1, 3, 1024, 1024]   [1, 64, 256, 256]    --                   True
│    └─Sequential (linknet_stem)         [1, 3, 1024, 1024]   [1, 64, 256, 256]    --                   True
│    │    └─Conv2d (0)                   [1, 3, 1024, 1024]   [1, 64, 512, 512]    9,408                True
│    │    └─BatchNorm2d (1)              [1, 64, 512, 512]    [1, 64, 512, 512]    128                  True
│    │    └─ReLU (2)                     [1, 64, 512, 512]    [1, 64, 512, 512]    --                   --
│    │    └─MaxPool2d (3)                [1, 64, 512, 512]    [1, 64, 256, 256]    --                   --
├─LinknetEncoderBlock (1)                [1, 64, 256, 256]    [1, 64, 128, 128]    --                   True
│    └─Sequential 

## 3. Let's now create the `LinknetDecoderBlock` with `nn.Module`

First, let's have the images back of the entire architecture, as well as de decoder block architecture, in order to have the numeration of the decoder blocks and the layer's requirements

<div style="text-align: center;">
    <img src="../docs/linknet_resnet18_architecture.png" width="350" style="margin-right: 10px;">
    <img src="../docs/linknet_resnet18_decoder-block.png" width="350">
</div>


So what are the constraints and requirements, these are shown from the following two diagrams:
- **Inputs/Outputs Dims**": The dimensions from the decoder blocks channels are defined in this table:

<img src="../docs/linknet_resnet18_inputs-outputs.png" width="350">

```python
x_input_decoder = encoder_4_output -> (B * 512 * 16 * 16)
x_input_decoder -> decoder_4 -> (B * 256 * 32 * 32)
out_linknet_decoder_4 -> decoder_3 -> (B * 128 * 64 * 64)
out_linknet_decoder_3 -> decoder_2 -> (B * 64 * 128 * 128)
out_linknet_decoder_2 -> decoder_1 -> (B * 64 * 256 * 256)

### 3.1 Let's define now the `LinknetDecoderBlock`

In [11]:
class LinknetDecoderBlock(nn.Module):
    '''
    Class that will simulate the Linknet Decoder blocks of linknnet architecture

    Args:
        m (int): number of input channels from linknet paper.
        n (int): number of output channels from linknet paper.
    '''

    def __init__(self, m:int, n:int) -> None:
        super().__init__()

        # Let's define the convolutional 1st block
        self.conv_block_1 = nn.Sequential(
        nn.Conv2d(in_channels=m, out_channels = int(m/4), kernel_size=(1, 1), stride = (1, 1), padding = (0, 0), bias = False),
        nn.BatchNorm2d(num_features=int(m/4), eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
        nn.ReLU())

        # Let's define the full convolutional transpose block
        # NOTE: In linknet paper they use "full" convolution, which are ConvTranspose2d in Pytorch
        # However, to avoid checkerboard artifacts, we will use nn.Upsample followed by nn.Conv2d.
        self.upsample_block = nn.Sequential(
            nn.Upsample(scale_factor=2, mode = 'bilinear', align_corners=True),
            nn.Conv2d(in_channels=int(m/4), out_channels=int(m/4), kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False),
            nn.BatchNorm2d(num_features=int(m/4), eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
            nn.ReLU()
        )

        # Let's define the convolutional 2nd block now
        self.conv_block_2 = nn.Sequential(
        nn.Conv2d(in_channels=int(m/4), out_channels = n, kernel_size=(1, 1), stride = (1, 1), padding = (0, 0), bias = False),
        nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
        nn.ReLU())

    def forward(self, x) -> torch.Tensor:
        x = self.conv_block_1(x)
        x = self.upsample_block(x)
        x = self.conv_block_2(x)
        return x

### 3.2 Summary of `linknetDecoder`, simulating decoder_3

In [13]:
from torchinfo import summary
# Let's one decoder block to see how it works (decoder block #3)
test_decoder_block = LinknetDecoderBlock(m = 256, n = 128)

summary(model = test_decoder_block,
        input_size=(1, 256, 32, 32), # (batch_size, num_channels, height, width) -> (B * 256 * 32 * 32)
        col_names = ["input_size", "output_size", "num_params", "trainable"],
        col_width = 20,
        row_settings = ["var_names"]
        )

Layer (type (var_name))                       Input Shape          Output Shape         Param #              Trainable
LinknetDecoderBlock (LinknetDecoderBlock)     [1, 256, 32, 32]     [1, 128, 64, 64]     --                   True
├─Sequential (conv_block_1)                   [1, 256, 32, 32]     [1, 64, 32, 32]      --                   True
│    └─Conv2d (0)                             [1, 256, 32, 32]     [1, 64, 32, 32]      16,384               True
│    └─BatchNorm2d (1)                        [1, 64, 32, 32]      [1, 64, 32, 32]      128                  True
│    └─ReLU (2)                               [1, 64, 32, 32]      [1, 64, 32, 32]      --                   --
├─Sequential (upsample_block)                 [1, 64, 32, 32]      [1, 64, 64, 64]      --                   True
│    └─Upsample (0)                           [1, 64, 32, 32]      [1, 64, 64, 64]      --                   --
│    └─Conv2d (1)                             [1, 64, 64, 64]      [1, 64, 64, 64]     

### 3.3 Let's test all the decoder blocks.

In [14]:
from torchinfo import summary
# Let's recreate the sequence of encoder blocks from linknet architecture with linknet stem
test_blocks = nn.Sequential(
    LinknetDecoderBlock(m = 512, n = 256),
    LinknetDecoderBlock(m = 256, n = 128),
    LinknetDecoderBlock(m = 128, n = 64),
    LinknetDecoderBlock(m = 64, n = 64)
)

summary(model = test_blocks,
        input_size=(1, 512, 16, 16), # (batch_size, num_channels, height, width) -> Encoder block 4 input
        col_names = ["input_size", "output_size", "num_params", "trainable"],
        col_width = 20,
        row_settings = ["var_names"]
        )

Layer (type (var_name))                  Input Shape          Output Shape         Param #              Trainable
Sequential (Sequential)                  [1, 512, 16, 16]     [1, 64, 256, 256]    --                   True
├─LinknetDecoderBlock (0)                [1, 512, 16, 16]     [1, 256, 32, 32]     --                   True
│    └─Sequential (conv_block_1)         [1, 512, 16, 16]     [1, 128, 16, 16]     --                   True
│    │    └─Conv2d (0)                   [1, 512, 16, 16]     [1, 128, 16, 16]     65,536               True
│    │    └─BatchNorm2d (1)              [1, 128, 16, 16]     [1, 128, 16, 16]     256                  True
│    │    └─ReLU (2)                     [1, 128, 16, 16]     [1, 128, 16, 16]     --                   --
│    └─Sequential (upsample_block)       [1, 128, 16, 16]     [1, 128, 32, 32]     --                   True
│    │    └─Upsample (0)                 [1, 128, 16, 16]     [1, 128, 32, 32]     --                   --
│    │    └─Conv2d

## 4. Reconstructer

<img src="../docs/linknet_resnet18_reconstructer.png" width="350">

Our input now is gonna be dimension `B * 64 * 256 * 256`, meaning that after passing these layers, we would have:

```python
x_input = out_linknet_decoder_1 -> (B * 64 * 256 * 256)
x_input -> full_conv_1 -> (B * 32 * 512 * 512)
full_conv_1_out -> conv -> (B * 32 * 512 * 512)
conv_out -> full_conv_2 -> (B * N * 1024 * 1024)
```

Where `N` is the desired number of channels out in our architecture! Since we got an image of size `B * Cin * Hin * Win` where `Cin = 3`, default will be `1024`; just as the DocLayNet data we got for our dataset.

In [15]:
class LinknetReconstructer(nn.Module):
    '''
    Class that will resemble the last Linknet reconstructer layer to upsample the desired mask

    Args:
        N (int): The number of desired out channels from the output
        m (int): number of input channels from linknet paper.
        n (int): number of output channels from linknet paper.

    '''
    def __init__(self, N:int = 1024, m:int = 64, n:int = 32):
        super().__init__()

        # Let's define the first upsample block
        self.upsample_block_1 = nn.Sequential(
        nn.Upsample(scale_factor=2, mode = 'bilinear', align_corners=True),
        nn.Conv2d(in_channels=m, out_channels=n, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False),
        nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
        nn.ReLU())

        # Let's define the convolutional final block
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_channels=n, out_channels=n, kernel_size=(3, 3), stride = (1, 1), padding=(1, 1), bias=False),
            nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
            nn.ReLU())
        
        # Let's define the final upsample layer
        # NOTE: The last Conv2d will output desired N channels, the paper says (2 x 2); but close inspection shows
        # that using (2 x 2) with padding (1, 1) will increase the output size by 1 pixel in height and width.
        # Therefore, we will use (3 x 3) kernel with padding (1, 1) to keep the same size after upsampling.

        self.upsample_block_2 = nn.Sequential(
        nn.Upsample(scale_factor=2, mode = 'bilinear', align_corners=True),
        nn.Conv2d(in_channels=n, out_channels=N, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False),
        nn.BatchNorm2d(num_features=N, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
        nn.ReLU())
    
    def forward(self, x) -> torch.Tensor:
        x = self.upsample_block_1(x)
        x = self.conv_block(x)
        x = self.upsample_block_2(x)
        return x

### 4.1 Summary of `linknetReconstructer`

In [16]:
from torchinfo import summary
# Let's one decoder block to see how it works (decoder block #4)
test_decoder_block = LinknetReconstructer(N=3, m=64, n = 32)

summary(model = test_decoder_block,
        input_size=(1, 64, 256, 256), # (batch_size, num_channels, height, width) -> (B * 64 * 256 * 256)
        col_names = ["input_size", "output_size", "num_params", "trainable"],
        col_width = 20,
        row_settings = ["var_names"]
        )

Layer (type (var_name))                       Input Shape          Output Shape         Param #              Trainable
LinknetReconstructer (LinknetReconstructer)   [1, 64, 256, 256]    [1, 3, 1024, 1024]   --                   True
├─Sequential (upsample_block_1)               [1, 64, 256, 256]    [1, 32, 512, 512]    --                   True
│    └─Upsample (0)                           [1, 64, 256, 256]    [1, 64, 512, 512]    --                   --
│    └─Conv2d (1)                             [1, 64, 512, 512]    [1, 32, 512, 512]    18,432               True
│    └─BatchNorm2d (2)                        [1, 32, 512, 512]    [1, 32, 512, 512]    64                   True
│    └─ReLU (3)                               [1, 32, 512, 512]    [1, 32, 512, 512]    --                   --
├─Sequential (conv_block)                     [1, 32, 512, 512]    [1, 32, 512, 512]    --                   True
│    └─Conv2d (0)                             [1, 32, 512, 512]    [1, 32, 512, 512]   

## 5 Final Step, let's create the entire network
Now that we have:
- `LinknetStem`
- `LinknetEncoder`
- `LinknetDecoder`
- `LinknetReconstructer`

Its time to use all of these classes and recreate the `LinknetModel`!

In [17]:
class LinknetModel(nn.Module):
    '''
    Class that simulates the full linknet architecture, using
    the rest of the blocks define before

    Args:
        Cin (int): number of input channels for the stem layer.
        N (int): number of output channels for the final reconstructer layer.
    '''

    def __init__(self, Cin:int = 3, N:int = 3) -> None:
        super().__init__()

        # Let's define the stem part
        self.stem = LinknetStem(m = Cin, n = 64)

        # Let's define each of the encoder blocks
        self.encoder_block_1 = LinknetEncoderBlock(m = 64, n = 64)
        self.encoder_block_2 = LinknetEncoderBlock(m = 64, n = 128)
        self.encoder_block_3 = LinknetEncoderBlock(m = 128, n = 256)
        self.encoder_block_4 = LinknetEncoderBlock(m = 256, n = 512)

        # Let's define each of the decoder blocls
        self.decoder_block_4 = LinknetDecoderBlock(m = 512, n = 256)
        self.decoder_block_3 = LinknetDecoderBlock(m = 256, n = 128)
        self.decoder_block_2 = LinknetDecoderBlock(m = 128, n = 64)
        self.decoder_block_1 = LinknetDecoderBlock(m = 64, n = 64)

        # Let's define the reconstructer part
        self.reconstructer = LinknetReconstructer(N=N, m=64, n=32)

    def forward(self, x) -> torch.Tensor:
        # Let's get the stem output
        x = self.stem(x)

        # Let's pass through each of the encoder blocks and store the outputs
        x1 = self.encoder_block_1(x)
        x2 = self.encoder_block_2(x1)
        x3 = self.encoder_block_3(x2)
        x4 = self.encoder_block_4(x3)

        # Let's pass through each of decoder blocks, while adding the skip connections
        x = self.decoder_block_4(x4) + x3
        x = self.decoder_block_3(x) + x2
        x = self.decoder_block_2(x) + x1

        # Finall, pass through the last decoder block and reconstructer
        x = self.decoder_block_1(x)
        x = self.reconstructer(x)
        return x

### 5.1 Summary with images of shape `(B * 3 * 1024 * 1024)`

In [18]:
from torchinfo import summary
# Let's one decoder block to see how it works (decoder block #4)
test_decoder_block = LinknetModel(Cin=3, N=1)

summary(model = test_decoder_block,
        input_size=(1, 3, 1024, 1024), # (batch_size, num_channels, height, width) -> (B * 64 * 256 * 256)
        col_names = ["input_size", "output_size", "num_params", "trainable"],
        col_width = 20,
        row_settings = ["var_names"]
        )

Layer (type (var_name))                       Input Shape          Output Shape         Param #              Trainable
LinknetModel (LinknetModel)                   [1, 3, 1024, 1024]   [1, 1, 1024, 1024]   --                   True
├─LinknetStem (stem)                          [1, 3, 1024, 1024]   [1, 64, 256, 256]    --                   True
│    └─Sequential (linknet_stem)              [1, 3, 1024, 1024]   [1, 64, 256, 256]    --                   True
│    │    └─Conv2d (0)                        [1, 3, 1024, 1024]   [1, 64, 512, 512]    9,408                True
│    │    └─BatchNorm2d (1)                   [1, 64, 512, 512]    [1, 64, 512, 512]    128                  True
│    │    └─ReLU (2)                          [1, 64, 512, 512]    [1, 64, 512, 512]    --                   --
│    │    └─MaxPool2d (3)                     [1, 64, 512, 512]    [1, 64, 256, 256]    --                   --
├─LinknetEncoderBlock (encoder_block_1)       [1, 64, 256, 256]    [1, 64, 128, 128]   

### 5.2 Summary with images of shape `(B * 3 * 512 * 512)`

In [19]:
from torchinfo import summary
# Let's one decoder block to see how it works (decoder block #4)
test_decoder_block = LinknetModel(Cin=3, N=1)

summary(model = test_decoder_block,
        input_size=(1, 3, 512, 512), # (batch_size, num_channels, height, width) -> (B * 64 * 256 * 256)
        col_names = ["input_size", "output_size", "num_params", "trainable"],
        col_width = 20,
        row_settings = ["var_names"]
        )

Layer (type (var_name))                       Input Shape          Output Shape         Param #              Trainable
LinknetModel (LinknetModel)                   [1, 3, 512, 512]     [1, 1, 512, 512]     --                   True
├─LinknetStem (stem)                          [1, 3, 512, 512]     [1, 64, 128, 128]    --                   True
│    └─Sequential (linknet_stem)              [1, 3, 512, 512]     [1, 64, 128, 128]    --                   True
│    │    └─Conv2d (0)                        [1, 3, 512, 512]     [1, 64, 256, 256]    9,408                True
│    │    └─BatchNorm2d (1)                   [1, 64, 256, 256]    [1, 64, 256, 256]    128                  True
│    │    └─ReLU (2)                          [1, 64, 256, 256]    [1, 64, 256, 256]    --                   --
│    │    └─MaxPool2d (3)                     [1, 64, 256, 256]    [1, 64, 128, 128]    --                   --
├─LinknetEncoderBlock (encoder_block_1)       [1, 64, 128, 128]    [1, 64, 64, 64]     